In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.melon.com/index.htm"
}

url = "https://www.melon.com/chart/index.htm"

response = requests.get(url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

rows = soup.select(
    "div.service_list_song.type02.d_song_list tbody tr"
)

data = []
song_ids = []

for row in rows:
    song_id = row.get("data-song-no")
    rank_tag = row.select_one("div.wrap.t_center span")
    title_tag = row.select_one("div.wrap_song_info span a")
    singer_tag = row.select_one("div.ellipsis.rank02 a")
    album_tag = row.select_one("div.ellipsis.rank03 a")

    if not all([
        song_id,
        rank_tag,
        title_tag,
        singer_tag,
        album_tag
    ]):
        continue

    rank = rank_tag.get_text(strip=True)
    title = title_tag.get_text(strip=True)
    singer = singer_tag.get_text(strip=True)
    album = album_tag.get_text(strip=True)

    song_ids.append(song_id)

    data.append({
        "곡 ID": song_id,
        "순위": rank,
        "곡 제목": title,
        "가수명": singer,
        "앨범명": album
    })

like_url = "https://www.melon.com/commonlike/getSongLike.json"
params = {
    "contsIds": ",".join(song_ids)
}
like_response = requests.get(
    like_url,
    params=params,
    headers=headers
)
like_response.raise_for_status()
like_data = like_response.json()
like_dict = {}

for item in like_data["contsLike"]:
    song_id = str(item["CONTSID"])
    like_count = item["SUMMCNT"]
    like_dict[song_id] = like_count

for song in data:
    song_id = song["곡 ID"]
    song["좋아요"] = like_dict.get(song_id, 0)

df = pd.DataFrame(data)
df = df.drop(columns=["곡 ID"])

file_name = "melon_chart.csv"

df.to_csv(
    file_name,
    index=False,
    encoding="utf-8-sig"
)

df

,순위,곡 제목,가수명,앨범명,좋아요
0,1,갑자기,아이오아이 (I.O.I),I.O.I 3rd MINI ALBUM [I.O.I : LOOP],56178
1,2,REDRED,CORTIS (코르티스),GREENGREEN,65069
2,3,LEMONADE,aespa,LEMONADE - The 2nd Album,40134
3,4,It′s Me,아일릿(ILLIT),MAMIHLAPINATAPAI,43582
4,5,LOVE ATTACK,RESCENE (리센느),SCENEDROME,84621
...,...,...,...,...,...
95,96,Soda Pop,KPop Demon Hunters Cast,KPop Demon Hunters (Soundtrack from the Netfli...,116732
96,97,KISS KISS KISS (Feat. 선우 (THE BOYZ)) (Prod. by...,NOWIMYOUNG (나우아임영),쇼미더머니 12 Episode 2,33111
97,98,I AM,IVE (아이브),I've IVE,238970
98,99,순간을 영원처럼,임영웅,IM HERO 2,49530
